<div class="alert alert-block alert-success" style="font-family: Times New Roman">
    <h4><strong>Laboratory Task 3</strong></h4>
<p style="font-family:Times New Roman; text-align:justify; font-size:15px">
    <b>Instruction:</b> Perform a forward and backward propagation in python using the inputs from <b>Laboratory Task 2</b>
</p>

```python
x = np.array([1, 0, 1])
y = np.array([1])

# use relu as the activation function.

# learning rate
lr = 0.001
```
</div>

### My thought process

This is the same 3 → 2 → 1 network I worked out in Laboratory Task 2 (same weights $w_{11}\dots w_{16}$, $w_{21}, w_{22}$, same biases $\theta_1,\theta_2,\theta_3$), but now I need to go a step further: after I get a prediction $\hat{y}$ from the **forward pass**, I have to run a **backward pass** to figure out how each weight should change to reduce the error, and then actually apply that update using the learning rate $lr = 0.001$ I was given.

**Choosing a loss function.** Since $\hat y$ is a real number coming out of ReLU (not a probability), I'll use the standard squared-error loss for regression-style outputs — it's also the same loss I already computed in Task 2:

$$L = \frac{1}{2}(y - \hat{y})^2$$

**My backprop plan (chain rule, one layer at a time):**

1. **Output layer error signal.**
   $$\frac{\partial L}{\partial \hat y} = \hat y - y, \qquad \delta_{O} = \frac{\partial L}{\partial \hat y}\cdot \text{ReLU}'(Z_O)$$
   where $\text{ReLU}'(z) = 1$ if $z>0$ else $0$.

2. **Gradients for the output-layer weights/bias.**
   $$\frac{\partial L}{\partial w_{2i}} = \delta_O \cdot H_i, \qquad \frac{\partial L}{\partial \theta_3} = \delta_O$$

3. **Propagate the error back to the hidden layer.**
   $$\delta_{H_i} = \delta_O \cdot w_{2i} \cdot \text{ReLU}'(Z_{H_i})$$

4. **Gradients for the hidden-layer weights/bias.**
   $$\frac{\partial L}{\partial w_{1ji}} = \delta_{H_i} \cdot x_j, \qquad \frac{\partial L}{\partial \theta_i} = \delta_{H_i}$$

5. **Gradient descent update** for every parameter $\theta$ (weight or bias):
   $$\theta \leftarrow \theta - lr \cdot \frac{\partial L}{\partial \theta}$$

I'll write this generically with small NumPy matrices instead of hard-coding each $w_{ij}$ separately — that way I can reuse the same `forward()`/`backward()` functions instead of repeating myself, and it's easier to double-check against the by-hand numbers from Task 2.


In [1]:
import numpy as np

np.set_printoptions(precision=6, suppress=True)

# ---- inputs given to me in the instructions ----
x = np.array([1, 0, 1])
y = np.array([1])

lr = 0.001  # learning rate

### Network parameters

Same weights and biases I used in Laboratory Task 2, just organized as NumPy matrices so I can use matrix multiplication instead of writing out every $w_{ij}$ by hand again.

In [2]:
# Hidden layer weights: rows = x1, x2, x3 | columns = H1, H2
W1 = np.array([
    [ 0.2, -0.3],   # w11, w12
    [ 0.4,  0.1],   # w13, w14
    [-0.5,  0.2],   # w15, w16
], dtype=float)
b1 = np.array([-0.4, 0.2], dtype=float)   # theta1, theta2

# Output layer weights: from H1, H2 -> O
W2 = np.array([[-0.3],
               [-0.2]], dtype=float)
b2 = np.array([0.1], dtype=float)          # theta3

print("W1 (hidden weights):\n", W1)
print("b1 (hidden biases):", b1)
print("W2 (output weights):\n", W2)
print("b2 (output bias):", b2)

W1 (hidden weights):
 [[ 0.2 -0.3]
 [ 0.4  0.1]
 [-0.5  0.2]]
b1 (hidden biases): [-0.4  0.2]
W2 (output weights):
 [[-0.3]
 [-0.2]]
b2 (output bias): [0.1]


### Forward pass

This is the same computation I did in Laboratory Task 2, just wrapped in a function this time so I can reuse the *pre-activation* values ($Z$) later when I need them for backprop, instead of recomputing them.

In [3]:
def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    # derivative of ReLU: 1 where z > 0, else 0
    return (z > 0).astype(float)

def forward(x, W1, b1, W2, b2):
    '''Runs one forward pass and returns everything I'll need for backprop.'''
    Z_hidden = x @ W1 + b1          # net input to hidden layer
    H = relu(Z_hidden)              # hidden activations

    Z_output = H @ W2 + b2          # net input to output layer
    y_hat = relu(Z_output)          # final prediction

    cache = {"Z_hidden": Z_hidden, "H": H, "Z_output": Z_output, "y_hat": y_hat}
    return y_hat, cache

y_hat, cache = forward(x, W1, b1, W2, b2)

print("Z_hidden:", cache["Z_hidden"])
print("H       :", cache["H"])
print("Z_output:", cache["Z_output"])
print("y_hat   :", y_hat)

loss = 0.5 * np.sum((y - y_hat) ** 2)
print("\nLoss L = 1/2(y - y_hat)^2 :", loss)

Z_hidden: [-0.7  0.1]
H       : [0.  0.1]
Z_output: [0.08]
y_hat   : [0.08]

Loss L = 1/2(y - y_hat)^2 : 0.4232


Good — these numbers match what I got in Laboratory Task 2 ($\hat y = 0.08$, loss $= 0.4232$), so I know my forward pass is still correct before I move on to backprop.

### Backward pass

Now I walk the chain rule backwards through the network, following the plan I laid out at the top: output layer first, then propagate the error signal $\delta$ back into the hidden layer.


In [4]:
def backward(x, y, W2, cache):
    '''Computes gradients for every weight/bias via backpropagation.'''
    H, Z_hidden = cache["H"], cache["Z_hidden"]
    Z_output, y_hat = cache["Z_output"], cache["y_hat"]

    # --- Output layer ---
    dL_dyhat = (y_hat - y)                      # dL/dy_hat, since L = 1/2(y-yhat)^2
    delta_output = dL_dyhat * relu_derivative(Z_output)   # delta_O

    grad_W2 = np.outer(H, delta_output)         # dL/dW2
    grad_b2 = delta_output                      # dL/db2 (theta3)

    # --- Hidden layer (error propagated backward through W2) ---
    delta_hidden = (delta_output @ W2.T) * relu_derivative(Z_hidden)  # delta_H

    grad_W1 = np.outer(x, delta_hidden)         # dL/dW1
    grad_b1 = delta_hidden                      # dL/db1 (theta1, theta2)

    grads = {"W1": grad_W1, "b1": grad_b1, "W2": grad_W2, "b2": grad_b2}
    return grads, delta_output, delta_hidden

grads, delta_output, delta_hidden = backward(x, y, W2, cache)

print("delta_output (output error signal):", delta_output)
print("delta_hidden (hidden error signal):", delta_hidden)
print()
print("grad_W1 (dL/dW1):\n", grads["W1"])
print("grad_b1 (dL/db1):", grads["b1"])
print("grad_W2 (dL/dW2):\n", grads["W2"])
print("grad_b2 (dL/db2):", grads["b2"])

delta_output (output error signal): [-0.92]
delta_hidden (hidden error signal): [0.    0.184]

grad_W1 (dL/dW1):
 [[0.    0.184]
 [0.    0.   ]
 [0.    0.184]]
grad_b1 (dL/db1): [0.    0.184]
grad_W2 (dL/dW2):
 [[-0.   ]
 [-0.092]]
grad_b2 (dL/db2): [-0.92]


A couple of things jump out at me here, and I think both trace back to ReLU:

- $\delta_{H_1} = 0$, because $Z_{H_1} = -0.7 < 0$, so $\text{ReLU}'(Z_{H_1}) = 0$ — this hidden unit was "off" during my forward pass, so no gradient flows through it at all. I remember this is called the **dying ReLU** problem.
- Only the weights feeding into $H_2$ ($w_{12}, w_{14}, w_{16}, \theta_2$) and the output layer end up with non-zero gradients. That makes sense to me — if a unit never fired, changing its incoming weights a tiny bit wouldn't have changed the output at all in this step, so there's nothing to update.


### Parameter update

Now I apply plain gradient descent with the learning rate I was given, $lr = 0.001$:

$$\theta \leftarrow \theta - lr \cdot \frac{\partial L}{\partial \theta}$$

In [5]:
W1_new = W1 - lr * grads["W1"]
b1_new = b1 - lr * grads["b1"]
W2_new = W2 - lr * grads["W2"]
b2_new = b2 - lr * grads["b2"]

print("Updated W1:\n", W1_new)
print("Updated b1 (theta1, theta2):", b1_new)
print("Updated W2:\n", W2_new)
print("Updated b2 (theta3):", b2_new)

Updated W1:
 [[ 0.2      -0.300184]
 [ 0.4       0.1     ]
 [-0.5       0.199816]]
Updated b1 (theta1, theta2): [-0.4       0.199816]
Updated W2:
 [[-0.3     ]
 [-0.199908]]
Updated b2 (theta3): [0.10092]


### Checking my work — did the update actually reduce the loss?

I want to make sure my update actually helped, not just that the arithmetic ran without errors. So I'll run a fresh forward pass using the *updated* weights and compare the new loss to the original loss from before backprop.

In [6]:
y_hat_new, _ = forward(x, W1_new, b1_new, W2_new, b2_new)
loss_new = 0.5 * np.sum((y - y_hat_new) ** 2)

print(f"y_hat before update : {y_hat[0]:.6f}   | loss before: {loss:.6f}")
print(f"y_hat after update  : {y_hat_new[0]:.6f}   | loss after : {loss_new:.6f}")
print(f"\nLoss decreased by   : {loss - loss_new:.8f}")

y_hat before update : 0.080000   | loss before: 0.423200
y_hat after update  : 0.081040   | loss after : 0.422244

Loss decreased by   : 0.00095584


### What I found

- My forward pass reproduces the same result I got in Laboratory Task 2: $\hat{y} = 0.08$, loss $= 0.4232$, so I know I carried the network over correctly.
- Backpropagation with the squared-error loss only gives me non-zero gradients for the parameters connected to $H_2$, the hidden unit that was actually "active." $H_1$'s parameters get zero gradient this step because ReLU clipped $H_1$ to 0 on the forward pass.
- After one gradient-descent step with $lr=0.001$, $\hat y$ moves slightly closer to $y=1$ and the loss goes down — which is exactly what I'd expect after a single iteration of training. It's a tiny step because the learning rate is small, but it's moving in the right direction, so I'm confident my backprop implementation is correct.
